In [ ]:
# 加载 Kaggle 数据集并进行数据预处理，包括生成候选区域。

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import torchvision.transforms as T
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import selectivesearch

# 数据集路径
DATASET_PATH = "/path/to/sixhky/open-images-bus-trucks"

# 加载标注数据
annotations_path = os.path.join(DATASET_PATH, "train-annotations-bbox.csv")
annotations = pd.read_csv(annotations_path)

# 过滤需要的类别（Bus 和 Truck）
annotations = annotations[annotations['LabelName'].isin(['/m/01bjv', '/m/07jdr'])]

# 映射类别到整数
class_mapping = {'/m/01bjv': 1, '/m/07jdr': 0}  # Bus 为 1，Truck 为 0
annotations['ClassID'] = annotations['LabelName'].map(class_mapping)

# 添加图像路径
image_dir = os.path.join(DATASET_PATH, "train")
annotations['ImagePath'] = annotations['ImageID'].apply(lambda x: os.path.join(image_dir, x + ".jpg"))

# 检查图像文件是否存在
annotations = annotations[annotations['ImagePath'].apply(lambda x: os.path.exists(x))]

# 划分训练集和测试集
train_annotations, test_annotations = train_test_split(annotations, test_size=0.2, random_state=42)

# 函数：选择性搜索生成候选区域
def generate_regions(image):
    """使用选择性搜索生成候选区域"""
    img_lbl, regions = selectivesearch.selective_search(image, scale=500, sigma=0.9, min_size=10)
    candidates = []
    for region in regions:
        x, y, w, h = region['rect']
        if w > 0 and h > 0:  # 确保宽高有效
            candidates.append((x, y, w, h))
    return candidates

# 函数：计算 IOU（Intersection over Union）
def compute_iou(box1, box2):
    """计算两个矩形框的 IOU"""
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2
    xi1 = max(x1, x2)
    yi1 = max(y1, y2)
    xi2 = min(x1 + w1, x2 + w2)
    yi2 = min(y1 + h1, y2 + h2)
    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    box1_area = w1 * h1
    box2_area = w2 * h2
    union_area = box1_area + box2_area - inter_area
    return inter_area / union_area

In [ ]:
# 创建一个自定义数据集，用于动态加载候选区域和标注数据。

In [ ]:
class RCNN_Dataset(Dataset):
    def __init__(self, annotations, image_size=(224, 224), iou_threshold=0.5):
        self.annotations = annotations
        self.image_size = image_size
        self.iou_threshold = iou_threshold
        self.transform = T.Compose([
            T.ToTensor(),
            T.Resize(self.image_size),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        row = self.annotations.iloc[index]
        image_path = row['ImagePath']
        image = cv2.imread(image_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # 生成候选区域
        regions = generate_regions(image)
        rois = []
        labels = []
        bbox_targets = []
        
        for region in regions:
            x, y, w, h = region
            roi = image[y:y+h, x:x+w]
            roi = cv2.resize(roi, self.image_size)
            roi = self.transform(roi).float()

            # 计算 IOU 并标注
            iou = compute_iou(region, (row['XMin'], row['YMin'], row['XMax'], row['YMax']))
            if iou > self.iou_threshold:
                label = row['ClassID']
                bbox_target = [
                    row['XMin'] - x,
                    row['YMin'] - y,
                    row['XMax'] - (x + w),
                    row['YMax'] - (y + h)
                ]
            else:
                label = 0  # 负样本
                bbox_target = [0, 0, 0, 0]

            rois.append(roi)
            labels.append(label)
            bbox_targets.append(bbox_target)
        
        return torch.stack(rois), torch.tensor(labels), torch.tensor(bbox_targets)

In [ ]:
# 在预训练的 VGG16 基础上添加分类和边界框回归层。

In [ ]:
import torch.nn as nn
import torchvision.models as models

class RCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(RCNN, self).__init__()
        # 加载预训练的 VGG16 模型
        vgg = models.vgg16(pretrained=True)
        self.features = vgg.features
        self.avgpool = vgg.avgpool
        self.flatten = nn.Flatten()
        
        # 分类分支
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, num_classes),  # 输出类别数
            nn.Sigmoid()
        )
        
        # 回归分支
        self.regressor = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096, 4)  # 输出边界框偏移量
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = self.flatten(x)
        class_output = self.classifier(x)
        bbox_output = self.regressor(x)
        return class_output, bbox_output

In [ ]:
# 训练 R-CNN 模型。

In [ ]:
from torch.optim import Adam

# 数据加载
train_dataset = RCNN_Dataset(train_annotations)
test_dataset = RCNN_Dataset(test_annotations)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

# 初始化模型
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RCNN(num_classes=2).to(device)

# 定义损失函数和优化器
criterion_class = nn.CrossEntropyLoss()
criterion_bbox = nn.MSELoss()
optimizer = Adam(model.parameters(), lr=1e-4)

# 训练循环
num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for rois, labels, bbox_targets in train_loader:
        rois = rois.to(device)
        labels = labels.to(device)
        bbox_targets = bbox_targets.to(device)

        # 前向传播
        class_output, bbox_output = model(rois)
        loss_class = criterion_class(class_output, labels)
        loss_bbox = criterion_bbox(bbox_output, bbox_targets)
        loss = loss_class + loss_bbox

        # 反向传播和优化
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

In [ ]:
# 在测试集上评估模型，并使用 NMS 消除重复的边界框。

In [ ]:
from torchvision.ops import nms

# 推理和 NMS
model.eval()
with torch.no_grad():
    for rois, labels, bbox_targets in test_loader:
        rois = rois.to(device)
        labels = labels.to(device)
        bbox_targets = bbox_targets.to(device)

        class_output, bbox_output = model(rois)
        scores, predicted_labels = torch.max(class_output, dim=1)

        # 应用 NMS
        keep = nms(bbox_output, scores, iou_threshold=0.5)
        final_boxes = bbox_output[keep]
        final_labels = predicted_labels[keep]
        print(f"Predicted Boxes: {final_boxes}, Labels: {final_labels}")